# Qiskit 2.x Examples

&copy; 2026 by [Damir Cavar](http://damir.cavar.me/)

The following examples work with [`qiskit` 2.x](https://docs.quantum.ibm.com/start/install). You have to create a file `secret.py` and declare the variable `api_key_ibm_q` in it:

    api_key_ibm_q = "92834787463912348790123749827349712394"

Replace the digits between the double quotes with your API token.

In Qiskit 2.x the old `qiskit-ibm-provider` package has been removed; access to IBM hardware now goes through `qiskit-ibm-runtime` (`QiskitRuntimeService` + the V2 primitives).

Prerequisites can be installed using the following code:

In [ ]:
!pip install -U "qiskit>=2.0"
!pip install -U "qiskit-ibm-runtime>=0.30"

# Random Bit Generator

The essential imports are listed below:

In [ ]:
from qiskit import QuantumRegister, ClassicalRegister, QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from secret import api_key_ibm_q

Connect to the IBM Quantum Platform runtime service. The legacy `ibm_quantum` channel was retired, so we use `ibm_quantum_platform`:

In [ ]:
service = QiskitRuntimeService(
    channel="ibm_quantum_platform",  # or "ibm_cloud"
    token=api_key_ibm_q,
)

Get a list of all available backends:

In [ ]:
backends = service.backends()

Print the list of backends:

In [ ]:
print(backends)

Pick a backend. In Qiskit 2.x it is common to select the least-busy operational device, but you can also request one by name:

In [ ]:
backend = service.least_busy(operational=True, simulator=False)
# Or, to request a specific device:
# backend = service.backend('ibm_brisbane')
print(backend.name)

We declare a quantum register and a classical register. The instantiated `circuit` object applies a Hadamard gate to all qubits (here set to 1, but set it to any number you might need). Then we apply a measure to the qubits and store the results in the classical register.

In [ ]:
q = QuantumRegister(1, 'q')
c = ClassicalRegister(1, 'c')
circuit = QuantumCircuit(q, c)
circuit.h(q)
circuit.measure(q, c)

In the following we transpile the circuit to the backend's ISA and run it through the V2 `Sampler` primitive. We request `shots=1` and set `memory=True` so we get the raw per-shot bitstring instead of just aggregated counts.

In [ ]:
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_qc = pm.run(circuit)

sampler = Sampler(mode=backend)
sampler.options.default_shots = 1
sampler.options.execution.memory = True
job = sampler.run([isa_qc])

Check the status on a regular basis to pull the result once the computation has finished. You can also use the Dashboard in IBM Quantum to see whether the computation has finished.

In [ ]:
status = backend.status()
print(status.operational)
print(status.pending_jobs)

Pull the result from the quantum endpoint once it is available. If you run this call before the result has been computed, the call will block until the result is available.

The V2 `Sampler` returns data keyed by the classical register name; for a single-shot job with `memory=True` we read the first (and only) bitstring:

In [ ]:
result = job.result()
pub_result = result[0]
bits = pub_result.data.c.get_bitstrings()
print('Random bit(s):', bits[0])

(C) 2026 by [Damir Cavar](http://damir.cavar.me/)